## Introducing Futures

Futures provide a nice way to reason about performing many operations in parallel, in an efficient and non-blocking way. 

A Future is a sort of a placeholder object that you can create for a result that *does not yet exist*. Generally, the result of the Future is computed concurrently and can be later collected. Composing concurrent tasks in this way tends to result in faster, asynchronous, non-blocking parallel code.

To use futures we must first import the `Future` type.

In [1]:
import concurrent.Future

import scala.util.Random

import concurrent.Future


import scala.util.Random


In simple terms, a `Future` is an object holding a value which may become available *at some point*. 

This value is usually the result of some other computation. The Future is *not completed* if the computation is not completed. If the computation has completed, then it returns a value, or an exception.

We can put a computation, e.g., a method inside a Future:

In [2]:
def longOperation =
    println("starting long operation")
    Thread.sleep(5000)
    val result = Random.nextInt(5)
    println(s"completed: $result")
    result

defined function longOperation

In [2]:
Future(longOperation)

-- Error: cmd2.sc:1:32 ---------------------------------------------------------
1 |val res2 = Future(longOperation)
  |                                ^
  |Cannot find an implicit ExecutionContext. You might add
  |an (implicit ec: ExecutionContext) parameter to your method.
  |
  |The ExecutionContext is used to configure how and on which
  |thread pools asynchronous tasks (such as Futures) will run,
  |so the specific ExecutionContext that is selected is important.
  |
  |If your application does not define an ExecutionContext elsewhere,
  |consider using Scala's global ExecutionContext by defining
  |the following:
  |
  |implicit val ec: scala.concurrent.ExecutionContext = scala.concurrent.ExecutionContext.global
  |
  |The following import might fix the problem:
  |
  |  import concurrent.ExecutionContext.Implicits.global
  |Compilation Failed

: 

However the comptation will fail, as the Future needs an execution context, as the error message says.

### Execution Context

The `ExecutionContext` specifies how and on which thread pool the Future code will execute. 


The `ExecutionContext` specifies how and on which thread pool the Future code will execute. 

We can create an `ExecutionContext` from `Executor` or `ExecutorService`. In this example we use a fixed thread pool of size 4. 

In [3]:
import scala.concurrent.ExecutionContext
import java.util.concurrent.Executors
val ec = ExecutionContext.fromExecutor(Executors.newFixedThreadPool(4))

import scala.concurrent.ExecutionContext

import java.util.concurrent.Executors

ec: ExecutionContextExecutor = scala.concurrent.impl.ExecutionContextImpl@52571203

Now with this execution Context we can call again the `longOperation` inside a `Future`:

In [5]:
Future(longOperation)(ec)

starting long operation


res4: Future[Int] = Success(value = 4)

The future will not be completed until the code finally returns a value. This is something that will *eventually* happen. Before that, the value will be still unavialable.


Now we can also call the long operation multiple times, each inside its own `Future`. Notice that they are asynchronous, so we do not need to wait until the end of each long operation before executing the next one. 

In [7]:
Future(longOperation)(ec)
Future(longOperation)(ec)
Future(longOperation)(ec)
Future(longOperation)(ec)

starting long operation
starting long operation
starting long operation
starting long operation


res6_0: Future[Int] = Success(value = 0)
res6_1: Future[Int] = Success(value = 2)
res6_2: Future[Int] = Success(value = 2)
res6_3: Future[Int] = Success(value = 0)

The execution context is an implicit parameter, so it can be passed uing a `given` declaration. 

Otherwise there is an ExecutionContext that we can use by default. 

It is a global built-in ExecutionContext, which uses `ForkJoinPool` with its parallelism level set to the number of available processors:

In [8]:
import concurrent.ExecutionContext.Implicits.global

import concurrent.ExecutionContext.Implicits.global


And now we do not need to explicitly indicate the execution context. As we are using the built-in global one, which is linked to the available number of processors. 

In [9]:
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)
Future(longOperation)

starting long operation
starting long operation
starting long operation
starting long operation
starting long operation
starting long operation
starting long operation
starting long operation


res8_0: Future[Int] = Success(value = 1)
res8_1: Future[Int] = Success(value = 0)
res8_2: Future[Int] = Success(value = 0)
res8_3: Future[Int] = Success(value = 3)
res8_4: Future[Int] = Success(value = 1)
res8_5: Future[Int] = Success(value = 3)
res8_6: Future[Int] = Success(value = 2)
res8_7: Future[Int] = Success(value = 1)
res8_8: Future[Int] = Success(value = 4)
res8_9: Future[Int] = Success(value = 2)
res8_10: Future[Int] = Success(value = 1)
res8_11: Future[Int] = Success(value = 3)
res8_12: Future[Int] = Success(value = 4)

## Futures vs Threads

We can identify certain differences while working with Threads and Futures.

- A Java Thread represents a thread of execution in an Java/Scala/JVM application.

- You can run code in parallel in a Thread, but a Thread does not have a return type.


## Futures vs Threads

We can identify certain differences while working with Threads and Futures.

- A Scala Future represents the result of an asynchronous computation, and has a return type.

- A Scala Future works with callback methods.

- A Scala Future can be composed, and has usual collection methods like map, flatMap, filter, etc.

- There is no guarantee that your Future’s callback method will be called on the same thread the future was run on.

## Future the functor

Future is a Functor, as an Option, and to be a type it needs a type parameter. For instance this future:

In [12]:
val f = Future(4)

f: Future[Int] = Success(value = 4)

is in reality a `Future[Int]`. We can spell it explicitly:

In [11]:
val f:Future[Int] = Future(5)

f: Future[Int] = Success(value = 5)

There are methods that can be called at any time (e.g., during the execution of the future), for example to check if it has been completed:

In [13]:
f.isCompleted

res12: Boolean = true

The value of the future can also be polled continuously. The value will be a `None` until the future is completed. Notice the type of the value: `Option[Try[Int]]`. 

In [14]:
f.value

res13: Option[Try[Int]] = Some(value = Success(value = 4))

In [15]:
f.value.get.isSuccess

res14: Boolean = true

It is easier to see how the value changes over time in the following example, where the future actually takes a while before resulting in a concrete value:

In [16]:
val f:Future[Int] = Future{Thread.sleep(10000); 5}

f: Future[Int] = Success(value = 5)

In [18]:
f.value

res17: Option[Try[Int]] = Some(value = Success(value = 5))

If we know the value of a future we can directly call it using `Future.successful`:

In [19]:
def getAFuture(i:Int):Future[Int]= {
    if i<5 then Future{Thread.sleep(5000); i}
    else Future.successful(i)
}    

defined function getAFuture

In [21]:
getAFuture(1)

res20: Future[Int] = Success(value = 1)

Futures can also end in a failure, with a Throwable (exception).

In [26]:
val f:Future[Int] = Future.failed(Exception("something went wrong"))

f: Future[Int] = Failure(exception = java.lang.Exception: something went wrong)

Futures come with other utilitarian mehtods, for instance for accessing the failure data (if any):

In [24]:
val f = Future(5)

f: Future[Int] = Success(value = 5)

In [27]:
f.failed.foreach{
     f => println(f)
}

java.lang.Exception: something went wrong